# Mechanism transfer on Hallem-Carlson (fly) — the clean stand

The M2OR holdout is confounded two ways: (a) **measurement structure** — M2OR is sparse and
which receptor x molecule pairs were assayed clusters by study, so a receptor's
"responds-to-class-C" label lives on a biased subset the model can exploit non-mechanistically;
(b) **general tuning** — a broadly-tuned receptor binds a bit of everything, so "responds to
C" is mostly tuning breadth, which the refined embedding already encodes.

The Hallem-Carlson matrix (fly, 24x110) fixes (a) **by construction**: every receptor is tested
on every odorant, so the measurement pattern is uniform — no assay-block confound, no sparsity.
We remove (b) explicitly by **residualising**: the target is each receptor's response to the
held-out class *above its own baseline responsiveness*. A model that only knows tuning cannot
predict that residual.

**Design.** For a chemically-coherent odorant class C: drop its columns, train the
full-coverage signed graph (**q=0**, 5 seeds) on the remaining odorants, get the
refined receptor embedding — which never saw a class-C odorant. Then predict each receptor's
held-out C-response from its neighbours in the embedding (leave-one-out kNN), and score the
correlation, **raw** (tuning-inflated) and **residualised** (specificity — the honest test).
We compare **three models**: **raw ESM** (structure only, sees no binding), **GNN+ESM** (both),
and **GNN one-hot** (function only — the same graph fed pure receptor identity instead of ESM,
so its geometry is learned entirely from binding), plus a **row-shuffle permutation null**. If a
graph beats ESM on the *residualised* score, it transferred a specific mechanism to an unseen
ligand class; the gap between GNN+ESM and GNN one-hot is what the structural prior adds. Small N (esp. HC=24) and strong receptor-correlation
(esp. CC) mean we read continuous correlations with per-seed CIs, not binary AUROC.


In [ ]:
%matplotlib inline
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")
from scipy.stats import t as _t

from orbind.regimes_ofm import ofm_pairs
from orbind.gnn_extractor import (
    GnnSignedExtractor, _build_universe, _mp_edges, _edge_index_dict, _train_one, MOL, PROT)

DATA   = ROOT / "data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT:", ROOT, "| device:", DEVICE)

In [ ]:
# ---- configuration -------------------------------------------------------
DATASETS = ["hc"]                 # Carey (mosquito), Hallem-Carlson (fly)
# full-coverage graph only (q=0); q99 dropped everywhere.
VARIANTS = {
    "full (q=0)":    dict(q=0.0,  criterion="coverage",          k_mode="coverage_quantile"),
}
SEEDS   = [42, 43, 44, 45, 46]
EPOCHS  = 900
K_NN    = 5
N_PERM  = 100            # row-shuffle permutations for the null band

SMARTS = {
    "carboxylic_acid": "[CX3](=O)[OX2H1]",
    "alcohol":         "[#6][OX2H]",
    "aldehyde":        "[CX3H1](=O)[#6]",
    "ketone":          "[#6][CX3](=O)[#6]",
    "ester":           "[#6][CX3](=O)[OX2H0][#6]",
    "aromatic":        "c1ccccc1",
}
MIN_MEMBERS = 6         # a class must have >= this many odorants to be held out
FOCUS   = ["carboxylic_acid"]   # the flagship (its picture + conclusion below); every other
                                # class gets the same three-way test in section 4

## 1 · Helpers: data, chemical classes, graph training, evaluation


In [ ]:
def load_ds(ds):
    p = ofm_pairs(ds)
    recs = sorted(p["receptor"].unique())
    ods  = sorted(p["inchikey"].unique())
    r_i  = {r: i for i, r in enumerate(recs)}
    o_i  = {o: j for j, o in enumerate(ods)}
    g = p.groupby(["receptor", "inchikey"], as_index=False)["label"].mean()
    R = np.full((len(recs), len(ods)), np.nan, np.float32)
    R[g["receptor"].map(r_i), g["inchikey"].map(o_i)] = g["label"].to_numpy()
    smi = p.drop_duplicates("inchikey").set_index("inchikey")["smiles"].to_dict()
    return p, recs, ods, R, smi

def classes_for(ods, smi):
    out = {}
    for cname, sm in SMARTS.items():
        q = Chem.MolFromSmarts(sm)
        members = [o for o in ods
                   if (m := Chem.MolFromSmiles(smi[o])) is not None and m.HasSubstructMatch(q)]
        if len(members) >= MIN_MEMBERS:
            out[cname] = set(members)
    return out

def train_refined(ds, p, variant_kw, seed, drop_iks, feat="esm"):
    """Signed graph trained on all pairs whose ODORANT is not in drop_iks (held-out class
    removed); returns {receptor: refined vector}, {receptor: raw ESM vector}.

    feat='esm'    -> ESM node features (structure + function).
    feat='onehot' -> one-hot receptor identity over the SAME receptors ESM covers, so the
                     receptor geometry is learned from binding alone (function only)."""
    ext = GnnSignedExtractor(name="cls",
        protein_path=str(DATA / f"embeddings/proteins/esm1b_650m_mean_{ds}.npz"),
        molecule_path=str(DATA / f"embeddings/molecules/gin_supervised_contextpred_{ds}.npz"),
        emit="prot", n_models=1, hidden=256, edge_threshold=0.0, task="regression",
        epochs=EPOCHS, **variant_kw)
    if feat == "onehot":
        recs = sorted(ext._proteins)
        eye  = np.eye(len(recs), dtype=np.float32)
        ext._proteins = {r: eye[i] for i, r in enumerate(recs)}
    idx = np.where((~p["inchikey"].isin(drop_iks)).to_numpy())[0]
    idx = idx[ext.covered(p, idx)]
    m2i, p2i, xm, xp = _build_universe(p, idx, ext._proteins, ext._molecules)
    pos, neg = _mp_edges(p, idx, m2i, p2i, ext.q, ext.criterion,
                         ext.edge_threshold, ext.k_mode, ext.task)
    pe, ne = _edge_index_dict(pos, neg)
    sub = p.iloc[idx]
    torch.manual_seed(seed)
    _, zp, _ = _train_one(ext._build_model, xm, xp, pe, ne,
                          sub["inchikey"].map(m2i).to_numpy(),
                          sub["receptor"].map(p2i).to_numpy(),
                          sub["label"].to_numpy(np.float32), ext._hp(seed), DEVICE, None)
    return {r: zp[i] for r, i in p2i.items()}, {r: xp[i].numpy() for r, i in p2i.items()}

def emb_sim(X):
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    return Xn @ Xn.T

def knn_pred(X, target, k):
    S = emb_sim(X).copy(); np.fill_diagonal(S, -np.inf)
    nn = np.argpartition(-S, k, axis=1)[:, :k]
    return np.stack([target[nn[i]].mean(0) for i in range(len(target))])

def col_corr(actual, pred):
    cs = []
    for o in range(actual.shape[1]):
        a, q = actual[:, o], pred[:, o]
        if a.std() > 1e-8 and q.std() > 1e-8:
            cs.append(np.corrcoef(a, q)[0, 1])
    return float(np.mean(cs)) if cs else np.nan

def perm_null(X, target, k, n=N_PERM):
    base = col_corr(target, knn_pred(X, target, k))
    rng = np.random.default_rng(0)
    null = []
    for _ in range(n):
        Xs = X[rng.permutation(len(X))]
        null.append(col_corr(target, knn_pred(Xs, target, k)))
    return base, float(np.mean(null)), float(np.std(null))

## 1b · See the class: response table + molecular skeletons

Before modelling, look at the carboxylic acids directly. The **complete** insect matrix makes
this crisp: reorder the odorants so the acids form a block on the left and sort receptors by
their acid response — if a distinct band of receptors lights up on the acid block but not
elsewhere, the class is functionally atypical (this is what `funcRedund` measured, now by
eye). Then the 2D structures: acids all carry the same `-C(=O)-O-H` hook, versus the nearest
non-acids and random odorants.


In [ ]:
from rdkit.Chem import Draw, AllChem, DataStructs
CVIZ = "carboxylic_acid"

for ds in DATASETS:
    p, recs, ods, R, smi = load_ds(ds)
    cl = classes_for(ods, smi)
    if CVIZ not in cl:
        print(ds, "no", CVIZ); continue
    o_i    = {o: j for j, o in enumerate(ods)}
    ac     = [o_i[o] for o in ods if o in cl[CVIZ]]
    rest   = [j for j in range(len(ods)) if j not in ac]
    rorder = np.argsort(-np.nan_to_num(np.nanmean(R[:, ac], axis=1)))   # receptors by acid response
    Mv     = R[np.ix_(rorder, ac + rest)]
    fig, ax = plt.subplots(figsize=(13, 4.8))
    im = ax.imshow(Mv, aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2, interpolation="nearest")
    ax.axvline(len(ac) - 0.5, color="black", lw=2.5)
    ax.set_title(f"{ds.upper()} responses (z-score); receptors sorted by acid response — acid block left of the line")
    ax.set_xlabel(f"odorants   [ {len(ac)} carboxylic acids | {len(rest)} others ]")
    ax.set_ylabel(f"{len(recs)} receptors")
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, fraction=0.02, label="response (z)"); fig.tight_layout(); plt.show()

    acids  = [o for o in ods if o in cl[CVIZ]]
    others = [o for o in ods if o not in cl[CVIZ]]
    def fp(ik):
        m = Chem.MolFromSmiles(smi[ik]); return AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048) if m else None
    def show_grid(iks, title, n=18):
        ms = [Chem.MolFromSmiles(smi[ik]) for ik in iks[:n]]; ms = [m for m in ms if m is not None]
        print(f"[{ds.upper()}] {title}"); display(Draw.MolsToGridImage(ms, molsPerRow=6, subImgSize=(150, 110)))
    fa   = [fp(o) for o in acids if fp(o) is not None]
    near = sorted(((o, max(DataStructs.BulkTanimotoSimilarity(fp(o), fa))) for o in others if fp(o) is not None),
                  key=lambda t: -t[1])
    rng = np.random.default_rng(0)
    show_grid(acids, f"carboxylic acids — held-out class ({len(acids)}); same -COOH hook")
    show_grid([o for o, _ in near], "nearest NON-acid odorants (top Tanimoto to acids)")
    show_grid(list(rng.choice(others, min(18, len(others)), replace=False)), "random odorants")

## 1c · The picture: actual acid responses vs all three models (full GNN)

Train the **full-coverage (q=0)** graph with all acids removed — once with ESM features, once
with one-hot identity — then reconstruct each receptor's acid response block from its neighbours
(leave-one-out kNN). Four panels, same z-score palette, receptors sorted by acid response:
**actual** | **GNN+ESM** (both) | **GNN one-hot** (function only) | **ESM** (structure only).
Panels that track the actual band carry the acid mechanism; the gap between the two graph panels
is what the structural prior adds on top of pure binding.


In [ ]:
for ds in DATASETS:
    p, recs, ods, R, smi = load_ds(ds)
    cl = classes_for(ods, smi)
    if CVIZ not in cl:
        continue
    ref_v, raw_v = train_refined(ds, p, VARIANTS["full (q=0)"], 42, cl[CVIZ])              # GNN(ESM)
    fun_v, _     = train_refined(ds, p, VARIANTS["full (q=0)"], 42, cl[CVIZ], feat="onehot")  # GNN(one-hot)
    order = [r for r in recs if r in ref_v and r in fun_v]; ri = [recs.index(r) for r in order]
    o_i = {o: j for j, o in enumerate(ods)}; ac = [o_i[o] for o in ods if o in cl[CVIZ]]
    A  = R[np.ix_(ri, ac)]                                       # actual acid responses
    Xr = np.stack([ref_v[r] for r in order]); Xe = np.stack([raw_v[r] for r in order])
    Xf = np.stack([fun_v[r] for r in order])
    def reconstruct(X):
        S = emb_sim(X).copy(); np.fill_diagonal(S, -np.inf)
        nn = np.argpartition(-S, K_NN, axis=1)[:, :K_NN]
        return np.stack([A[nn[i]].mean(0) for i in range(len(order))])
    P_our, P_fun, P_esm = reconstruct(Xr), reconstruct(Xf), reconstruct(Xe)
    ro = np.argsort(-np.nan_to_num(A.mean(1)))
    panels = [(A, "actual"), (P_our, "GNN+ESM (both)"),
              (P_fun, "GNN one-hot (function)"), (P_esm, "ESM (structure)")]
    fig, axes = plt.subplots(1, 4, figsize=(19, 5), sharey=True)
    for ax, (Mx, ttl) in zip(axes, panels):
        ax.imshow(Mx[ro], aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2, interpolation="nearest")
        ax.set_title(f"{ds.upper()} - {ttl}", fontsize=10); ax.set_xlabel(f"{len(ac)} acids")
        ax.set_xticks([]); ax.set_yticks([])
    axes[0].set_ylabel(f"{len(order)} receptors (sorted by acid response)")
    fig.tight_layout(); plt.show()

## 2 · Run the holdout: per dataset x class x variant x seed

For each held-out class we build the held-out response block `Ymat` (receptors x class
odorants) and its residual `resid = Ymat - baseline`, where baseline is the receptor's mean
response over the *retained* odorants (its general responsiveness). We score each embedding
on both.


In [ ]:
rows = []
null_rows = []
for ds in DATASETS:
    p, recs, ods, R, smi = load_ds(ds)
    cls = {c: v for c, v in classes_for(ods, smi).items() if c in FOCUS}
    print(f"\n=== {ds.upper()} ===  {len(recs)} receptors x {len(ods)} odorants | "
          f"classes: {{{', '.join(f'{c}:{len(v)}' for c, v in cls.items())}}}")
    o_i = {o: j for j, o in enumerate(ods)}
    for cname, iks in cls.items():
        cols  = [o_i[o] for o in ods if o in iks]
        keep  = [j for j in range(len(ods)) if j not in cols]

        def targets_for(order):
            ri    = [recs.index(r) for r in order]
            Ymat  = np.nan_to_num(R[np.ix_(ri, cols)])
            base  = np.nan_to_num(np.nanmean(R[np.ix_(ri, keep)], axis=1, keepdims=True))
            return Ymat, Ymat - base

        first = True
        for feat, tag in [("esm", "GNN+ESM"), ("onehot", "GNN one-hot")]:
            for vname, vkw in VARIANTS.items():
                for seed in SEEDS:
                    ref, raw = train_refined(ds, p, vkw, seed, iks, feat=feat)
                    order = [r for r in recs if r in ref]
                    Ymat, resid = targets_for(order)
                    Xr = np.stack([ref[r] for r in order])
                    if first:   # raw ESM baseline + null: same for every graph, take it once
                        Xe = np.stack([raw[r] for r in order])
                        rows.append(dict(ds=ds, cls=cname, emb="raw ESM", feat="esm", variant="-", seed="-",
                                         raw=col_corr(Ymat, knn_pred(Xe, Ymat, K_NN)),
                                         spec=col_corr(resid, knn_pred(Xe, resid, K_NN))))
                        _, nm, ns = perm_null(Xe, resid, K_NN)
                        null_rows.append(dict(ds=ds, cls=cname, null_mean=nm, null_sd=ns))
                        first = False
                    rows.append(dict(ds=ds, cls=cname, emb=f"{tag} {vname}", feat=feat, variant=vname, seed=seed,
                                     raw=col_corr(Ymat, knn_pred(Xr, Ymat, K_NN)),
                                     spec=col_corr(resid, knn_pred(Xr, resid, K_NN))))
                sub = [r["spec"] for r in rows if r["ds"] == ds and r["cls"] == cname
                       and r["feat"] == feat and r["variant"] == vname]
                esm = [r["spec"] for r in rows if r["ds"] == ds and r["cls"] == cname and r["emb"] == "raw ESM"][0]
                print(f"  {cname:16} {tag:12} {vname:14} spec r={np.nanmean(sub):+.3f} (ESM {esm:+.3f})")
res  = pd.DataFrame(rows)
null = pd.DataFrame(null_rows)

## 3 · Results — specificity (residualised) vs raw

`raw` = predict held-out response with tuning left in (inflated; refined wins mostly because it
knows tuning breadth). `spec` = predict the residual (specific mechanism, tuning removed) — the
honest test. Bars show the residualised score; the dotted line is the permutation null.


In [ ]:
def ci(x):
    x = np.asarray([v for v in x if np.isfinite(v)], float)
    if len(x) < 2: return (np.nanmean(x) if len(x) else np.nan, np.nan)
    return x.mean(), _t.ppf(.975, len(x) - 1) * x.std(ddof=1) / len(x) ** .5

# the three models: ESM structure-only, GNN+ESM (both), GNN one-hot (function only)
MODELS = [("raw ESM", None, None)]
for tag, feat in [("GNN+ESM", "esm"), ("GNN one-hot", "onehot")]:
    for vname in VARIANTS:
        MODELS.append((f"{tag} {vname}", feat, vname))

def model_spec(ds, c, feat, vname, metric):
    if feat is None:
        return res[(res.ds==ds)&(res.cls==c)&(res.emb=="raw ESM")][metric].iloc[0], 0.0
    return ci(res[(res.ds==ds)&(res.cls==c)&(res.feat==feat)&(res.variant==vname)][metric])

# summary table (both raw and spec)
sm = []
for (ds, cname), _ in res.groupby(["ds", "cls"]):
    row = {"ds": ds, "class": cname}
    for metric in ["raw", "spec"]:
        for label, feat, vname in MODELS:
            m, _ = model_spec(ds, cname, feat, vname, metric)
            row[f"{label} {metric}"] = m
    sm.append(row)
sm = pd.DataFrame(sm)
display(sm.style.format({c: "{:+.3f}" for c in sm.columns if c not in ("ds", "class")}).hide(axis="index"))

palette = {"raw ESM": "#888888",
           "GNN+ESM q99 (titular)": "#2f5c9e", "GNN+ESM full (q=0)": "#7aa0d4",
           "GNN one-hot q99 (titular)": "#b5651d", "GNN one-hot full (q=0)": "#e8a15c"}
for ds in DATASETS:
    classes = sm[sm.ds == ds]["class"].tolist()
    if not classes: continue
    labels = [m[0] for m in MODELS]
    fig, ax = plt.subplots(figsize=(1.9*len(classes)+6, 4.3))
    x = np.arange(len(classes)); w = 0.8 / len(labels)
    for j, (label, feat, vname) in enumerate(MODELS):
        vals, errs = [], []
        for c in classes:
            m, h = model_spec(ds, c, feat, vname, "spec"); vals.append(m); errs.append(h)
        ax.bar(x + (j-(len(labels)-1)/2)*w, vals, w, yerr=errs, capsize=2,
               label=label, color=palette.get(label, "#999999"))
    for xi, c in enumerate(classes):
        nm = null[(null.ds==ds)&(null.cls==c)]["null_mean"].iloc[0]
        ax.plot([xi-0.45, xi+0.45], [nm, nm], ls=":", c="gray", lw=1)
    ax.axhline(0, c="k", lw=.6)
    ax.set_xticks(x); ax.set_xticklabels(classes)
    ax.set_ylabel(f"specificity: held-out C-response corr (LOO {K_NN}-NN, tuning removed)")
    ax.set_title(f"{ds.upper()} — structure vs function vs both, on an UNSEEN class (dotted = null)")
    ax.legend(frameon=False, fontsize=8, ncol=2); fig.tight_layout(); plt.show()

**Reading it.** On the **raw** metric refined beats ESM easily — but that is largely tuning
breadth, which we already knew the graph encodes; it is *not* the mechanism claim. The
**specificity** bars are the real test: predicting a receptor's held-out class response *above
its own baseline*, a signal tuning cannot supply. If refined clears both the raw-ESM bar **and**
the permutation null there, the receptor embedding carries a specific, transferable binding
mechanism — demonstrated on a complete matrix (no measurement confound) with tuning explicitly
removed. Where refined only matches ESM/null on specificity, the honest conclusion is that the
transferable signal is generic responsiveness, not a class-specific mechanism. Read Hallem-Carlson with
care: with only 24 receptors, strong cross-correlation inflates absolute
correlations and shrinks the effective N — the ESM baseline and the null are what keep the
comparison fair.


## 4 · The other odorant classes — same three-way test (full GNN)

The carboxylic acid above is the flagship. Now run the **same** structure/function/both
comparison for every other chemical class with enough odorants (`MIN_MEMBERS`), still the
full-coverage graph, five seeds, residualised specificity + permutation null. The acid rows
already in `res`/`null` are reused, so nothing is recomputed for it.


In [ ]:
rows_o, null_o = [], []
for ds in DATASETS:
    p, recs, ods, R, smi = load_ds(ds)
    others = {c: v for c, v in classes_for(ods, smi).items() if c not in FOCUS}
    o_i = {o: j for j, o in enumerate(ods)}
    print(f"=== {ds.upper()} other classes: {{{', '.join(f'{c}:{len(v)}' for c, v in others.items())}}} ===")
    for cname, iks in others.items():
        cols = [o_i[o] for o in ods if o in iks]
        keep = [j for j in range(len(ods)) if j not in cols]

        def targets_for(order):
            ri   = [recs.index(r) for r in order]
            Ymat = np.nan_to_num(R[np.ix_(ri, cols)])
            base = np.nan_to_num(np.nanmean(R[np.ix_(ri, keep)], axis=1, keepdims=True))
            return Ymat, Ymat - base

        first = True
        for feat, tag in [("esm", "GNN+ESM"), ("onehot", "GNN one-hot")]:
            for seed in SEEDS:
                ref, raw = train_refined(ds, p, VARIANTS["full (q=0)"], seed, iks, feat=feat)
                order = [r for r in recs if r in ref]
                Ymat, resid = targets_for(order)
                Xr = np.stack([ref[r] for r in order])
                if first:
                    Xe = np.stack([raw[r] for r in order])
                    rows_o.append(dict(ds=ds, cls=cname, emb="raw ESM", feat="esm", variant="-", seed="-",
                                       raw=col_corr(Ymat, knn_pred(Xe, Ymat, K_NN)),
                                       spec=col_corr(resid, knn_pred(Xe, resid, K_NN))))
                    _, nm, ns = perm_null(Xe, resid, K_NN)
                    null_o.append(dict(ds=ds, cls=cname, null_mean=nm, null_sd=ns)); first = False
                rows_o.append(dict(ds=ds, cls=cname, emb=f"{tag} full (q=0)", feat=feat, variant="full (q=0)",
                                   seed=seed, raw=col_corr(Ymat, knn_pred(Xr, Ymat, K_NN)),
                                   spec=col_corr(resid, knn_pred(Xr, resid, K_NN))))
            sub = [r["spec"] for r in rows_o if r["ds"]==ds and r["cls"]==cname
                   and r["feat"]==feat and r["variant"]=="full (q=0)"]
            print(f"  {cname:16} {tag:12} spec r={np.nanmean(sub):+.3f}")

res_all  = pd.concat([res,  pd.DataFrame(rows_o)], ignore_index=True)
null_all = pd.concat([null, pd.DataFrame(null_o)], ignore_index=True)
CLASSES_ALL = FOCUS + sorted(set(res_all["cls"]) - set(FOCUS))
print("\nclasses in the comparison:", CLASSES_ALL)

## 5 · The comparison across all classes

Every held-out class on the x-axis, three specificity bars each — **ESM** (structure only),
**GNN+ESM** (both), **GNN one-hot** (function only) — with the dotted permutation null per class.
Bars clear of both the ESM bar and the null are class-specific mechanism transfer; bars hugging
the null are honest negatives (only generic responsiveness there). The acid should stand out; the
panel shows how special that is.


In [ ]:
MODELS2  = [("raw ESM", None), ("GNN+ESM full (q=0)", "esm"), ("GNN one-hot full (q=0)", "onehot")]
palette2 = {"raw ESM": "#888888", "GNN+ESM full (q=0)": "#2f5c9e", "GNN one-hot full (q=0)": "#b5651d"}

def spec_all(ds, c, feat):
    if feat is None:
        return res_all[(res_all.ds==ds)&(res_all.cls==c)&(res_all.emb=="raw ESM")]["spec"].iloc[0], 0.0
    return ci(res_all[(res_all.ds==ds)&(res_all.cls==c)&(res_all.feat==feat)&(res_all.variant=="full (q=0)")]["spec"])

# summary table (specificity)
tbl = []
for ds in DATASETS:
    for c in [c for c in CLASSES_ALL if c in set(res_all[res_all.ds==ds]["cls"])]:
        row = {"ds": ds, "class": c}
        for label, feat in MODELS2: row[label] = spec_all(ds, c, feat)[0]
        row["null"] = null_all[(null_all.ds==ds)&(null_all.cls==c)]["null_mean"].iloc[0]
        tbl.append(row)
tbl = pd.DataFrame(tbl)
display(tbl.style.format({c: "{:+.3f}" for c in tbl.columns if c not in ("ds", "class")}).hide(axis="index"))

for ds in DATASETS:
    classes = [c for c in CLASSES_ALL if c in set(res_all[res_all.ds==ds]["cls"])]
    if not classes: continue
    labels = [m[0] for m in MODELS2]
    fig, ax = plt.subplots(figsize=(1.9*len(classes)+5, 4.5))
    x = np.arange(len(classes)); w = 0.8 / len(labels)
    for j, (label, feat) in enumerate(MODELS2):
        vals = [spec_all(ds, c, feat)[0] for c in classes]
        errs = [spec_all(ds, c, feat)[1] for c in classes]
        ax.bar(x + (j-(len(labels)-1)/2)*w, vals, w, yerr=errs, capsize=2, label=label, color=palette2[label])
    for xi, c in enumerate(classes):
        nm = null_all[(null_all.ds==ds)&(null_all.cls==c)]["null_mean"].iloc[0]
        ax.plot([xi-0.45, xi+0.45], [nm, nm], ls=":", c="gray", lw=1)
    ax.axhline(0, c="k", lw=.6)
    ax.set_xticks(x); ax.set_xticklabels(classes, rotation=15, ha="right")
    ax.set_ylabel(f"specificity (LOO {K_NN}-NN, tuning removed)")
    ax.set_title(f"{ds.upper()} — mechanism transfer across odorant classes (dotted = null)")
    ax.legend(frameon=False, ncol=2); fig.tight_layout(); plt.show()

**Reading the panel.** The complete matrix and the residualised target make this the honest
cross-class test: a class where GNN+ESM (or one-hot) stands clear of the null and the ESM bar
transferred a *specific* mechanism the graph never trained on; a class where all three sit near
the null is a real negative — no class-specific structure to recover there, only tuning. The acid
standing out against several flatter classes is exactly what makes it a mechanism result and not
a method artefact; the GNN+ESM vs one-hot gap says whether sequence was needed on top of binding.
